In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from tqdm import tqdm

In [ ]:
def read_pmids_from_file(filename):
    """
    Reads PubMed IDs from a file and returns them as a list.
    """
    with open(filename, 'r') as file:
        pmids = [line.strip() for line in file]
    return pmids

pmids = read_pmids_from_file('pmids.txt')

In [ ]:
len(pmids)

In [ ]:
def splitList(input,chunkSize):
    return [input[i:i+chunkSize] for i in range(0,len(input),chunkSize)]

In [ ]:
for i in range(5):
    if i == 1:
        continue
    print(i)

In [ ]:
outputDf = pd.DataFrame(columns=['PMID','TITLE','YEAR','ABSTRACT'])
PMIDS = []
TITLES = []
YEARS = []
ABSTRACTS = []
pmids = splitList(pmids,100)
for id in tqdm(pmids):
    api_key = '20580ac1e3b1d871116c14f1e37049233d08'
    params = {
        'db': 'pubmed',
        'id': id,
        'retmode': 'xml',
        'api_key':api_key
    }
    base_url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?'
    response = requests.get(base_url, params)
    root = ET.fromstring(response.content)
    indx = ET.tostring(root, encoding='unicode')
    if response.status_code == 200:
        articles = root.findall('.//PubmedArticle')
        for article in articles:
            pmid = article.findall('.//PMID')
            # if only one pmid in the article
            PMIDS.append(pmid[0].text)
            try:
                date = article.findall('.//DateCompleted')
                #print(date)
                date = date[0]
                year = date.findall('.//Year')
                YEARS.append(year[0].text)
            except:
                year = None
                YEARS.append(year)
            
            title = article.findall('.//ArticleTitle')
            TITLES.append(title[0].text)
            abstract = article.findall('.//Abstract')
            if len(abstract)==0:
                ABSTRACTS.append('No abstract available')
            else:
                #print(abstract)
                abstract = abstract[0]
                abstractText = abstract.findall('.//AbstractText')
                if len(abstractText) >1:
                    #if multiple abstract sections
                    abText = []
                    for abstracttext in abstractText:
                        try:
                            label = abstracttext.get('Label')
                            if label is not None:
                                abText.append(str(label+':'+abstracttext.text))
                            else:
                                abText.append(abstracttext.text)
                        except:
                            abText.append('')
                    outputText = ''.join(abText)
                    ABSTRACTS.append(str(outputText))
                else:
                    # if only one abstract section
                    #print(abstractText)
                    ABSTRACTS.append(str(abstractText[0].text))                
    else:
        print(indx)

In [ ]:
print(len(PMIDS))
print(len(YEARS))
print(len(TITLES))
print(len(ABSTRACTS))

In [ ]:
outputDf['PMID'] = PMIDS
outputDf['TITLE'] = TITLES
outputDf['YEAR'] = YEARS
outputDf['ABSTRACT'] = ABSTRACTS

In [ ]:
outputDf.head(10)

In [ ]:
outputDf.to_csv('Data/head and neck cancer query abstracts.csv')

In [ ]:
outputDf

In [ ]:
outputDf[outputDf['YEAR'] > '2020']

### Data viewing

In [ ]:
outputDf = pd.read_csv('Data/head and neck cancer query abstracts.csv')

In [ ]:
### plot visualization of the number of articles per year, want my years to not be decimal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
### ensure the year column does not have non-finite values or NA
outputDf['YEAR'] = pd.to_numeric(outputDf['YEAR'], errors='coerce')
outputDf['YEAR'] = outputDf['YEAR'].fillna(0)  # Fill NaN with 0
outputDf['YEAR'] = outputDf['YEAR'].astype(int)
outputDf['YEAR'] = outputDf['YEAR'].astype(str)
# Plotting the number of articles per year
plt.figure(figsize=(20, 10))
### Count the number of articles per year and plot
### drop the year column that is 0
outputDf_graph = outputDf[outputDf['YEAR'] != '0']
outputDf_graph['YEAR'].value_counts().sort_index().plot(kind='bar')
plt.title('Number of Articles per Year')
plt.xlabel('Year')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
### plot histogram of the number of articles per year, increment into 5 year bins
outputDf['YEAR'] = pd.to_numeric(outputDf['YEAR'], errors='coerce')
outputDf['YEAR'].dropna(inplace=True)
plt.figure(figsize=(10, 6))
outputDf['YEAR'].hist(bins=range(1965, 2030, 5), edgecolor='black')
plt.title('Histogram of Articles per 5-Year Bins')
plt.xlabel('Year')
plt.ylabel('Number of Articles')
plt.xticks(range(1965, 2030, 5))
plt.show()

In [ ]:
### bucket the years into increments of 5 years
outputDf['YEAR_BUCKET'] = (outputDf['YEAR'] // 5) * 5
outputDf['YEAR_BUCKET'] = outputDf['YEAR_BUCKET'].fillna(-1).astype(int)
outputDf['YEAR_BUCKET'] = outputDf['YEAR_BUCKET'].astype(int)
plt.figure(figsize=(10, 6))
outputDf_graph = outputDf[(outputDf['YEAR_BUCKET'] != -1) & (outputDf['YEAR_BUCKET'] != 0)]
outputDf_graph['YEAR_BUCKET'].value_counts().sort_index().plot(kind='bar')
plt.title('Number of Articles per 5-Year Bucket')
plt.xlabel('Year Bucket')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
### percentage of articles after year 2000
outputDf['YEAR'] = pd.to_numeric(outputDf['YEAR'], errors='coerce')
percentage_after_2000 = (outputDf[outputDf['YEAR'] > 2000].shape[0] / outputDf.shape[0]) * 100
print(f"Percentage of articles after year 2000: {percentage_after_2000:.2f}%")